In [1]:
import pandas as pd

In [2]:
import numpy as np

In [50]:
import altair as alt

In [4]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [5]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [6]:
kar = pd.read_excel("pakistan_aqi_weather.xlsx", sheet_name = "Karachi")

In [7]:
print(kar["time"].dtype)

datetime64[ns]


In [8]:
kar.set_index('time', inplace=True)

In [9]:
kar = kar[[
    'pm2_5',
    'temperature_2m',
    'relative_humidity_2m',
    'pressure_msl',
    'wind_speed_10m',
    'rain',
    'cloud_cover'
]]

In [10]:
kar.fillna(method='ffill', inplace=True)

C:\Users\HP\AppData\Local\Temp\ipykernel_5904\2536592973.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  kar.fillna(method='ffill', inplace=True)


In [11]:
kar_monthly = kar.resample('M').mean()

C:\Users\HP\AppData\Local\Temp\ipykernel_5904\1344153399.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  kar_monthly = kar.resample('M').mean()


In [12]:
kar_reset = kar_monthly.reset_index()

In [13]:
print(kar_reset.head())

        time      pm2_5  temperature_2m  relative_humidity_2m  pressure_msl  \
0 2022-08-31  33.676912       27.910945             85.220390   1000.442279   
1 2022-09-30  27.695278       28.169861             77.730556   1005.597639   
2 2022-10-31  37.135349       27.538038             71.806452   1011.032661   
3 2022-11-30  49.267639       25.253056             70.451389   1014.382639   
4 2022-12-31  57.456452       21.436156             52.735215   1015.538978   

   wind_speed_10m      rain  cloud_cover  
0       18.050225  0.488756    85.838081  
1       19.443889  0.006111    41.331944  
2       13.859274  0.000538    10.745968  
3       11.773194  0.000972    17.716667  
4       12.389785  0.000000    10.173387  


In [14]:
alt.data_transformers.disable_max_rows()


chart = alt.Chart(kar_reset).mark_line().encode(
    x=alt.X('time:T', title='Date'),
    y=alt.Y('pm2_5:Q', title='PM2.5'),
    tooltip=['Date:T', 'pm2_5:Q']
).properties(
    title='5-Year PM2.5 Trend',
    width=800,
    height=400
).interactive()

chart

alt.Chart(...)

In [15]:
print(kar.index.min())
print(kar.index.max())

print(kar.shape)

2022-08-04 05:00:00
2026-05-18 23:00:00
(33211, 7)


In [16]:
train_size = int(len(kar) * 0.78)   # ~36 months 

train = kar.iloc[:train_size]
test = kar.iloc[train_size:]

In [27]:
train_monthly = train.resample('ME').mean()

In [28]:
train_monthly_plot = train_monthly.reset_index()

In [29]:
# # train_reset = train.reset_index()
# # test_reset = test.reset_index()

# #Convert pm2_5 to numeric
# train_reset['pm2_5'] = pd.to_numeric(train_reset['pm2_5'], errors='coerce')
# # test_reset['pm2_5'] = pd.to_numeric(test_reset['pm2_5'], errors='coerce')

# # TRAINING DATA GRAPH


train_chart = alt.Chart(train_monthly_plot).mark_line(color = 'red').encode(
    x=alt.X('time:T', title='Date'),
    y=alt.Y('pm2_5:Q', title='PM2.5'),
    tooltip=['index:T', 'pm2_5:Q']
).properties(
    title='AQI - Training Data (36 Months)',
    width=800,
    height=400
).interactive()

train_chart

alt.Chart(...)

In [30]:
test_monthly = test.resample('ME').mean()

In [31]:
train_monthly_plot = test_monthly.reset_index()

In [32]:
test_chart = alt.Chart(train_monthly_plot).mark_line(
    color='red'
).encode(
    x=alt.X('time:T', title='Date'),
    y=alt.Y('pm2_5:Q', title='PM2.5'),
    tooltip=['index:T', 'pm2_5:Q']
).properties(
    title='AQI - Test Data (10 Months)',
    width=800,
    height=400
).interactive()

test_chart

alt.Chart(...)

In [63]:
print("Full range:", kar.index.min(), "to", kar.index.max())
print("Train range:", train.index.min(), "to", train.index.max())
print("Test range:", test.index.min(), "to", test.index.max())

Full range: 2022-08-04 05:00:00 to 2026-05-18 23:00:00
Train range: 2022-08-04 05:00:00 to 2025-07-18 12:00:00
Test range: 2025-07-18 13:00:00 to 2026-05-18 23:00:00


# The study focuses on forecasting PM2.5 concentration as it is one of the most significant contributors to air quality variations and exhibits strong seasonal patterns suitable for SARIMA modeling.

In [ ]:
kar_monthly = kar.resample('ME').mean()

In [ ]:
train = kar_monthly.iloc[:36]
test = kar_monthly.iloc[36:]

In [ ]:
y_train = train['pm2_5']
X_train = train[['temperature_2m', 'relative_humidity_2m', 'pressure_msl', 'wind_speed_10m', 'rain', 'cloud_cover']]

y_test = test['pm2_5']
X_test = test[['temperature_2m', 'relative_humidity_2m', 'pressure_msl', 'wind_speed_10m', 'rain', 'cloud_cover']]

In [ ]:
model = SARIMAX(y_train,
                exog=X_train,
                order=(1,0,1),
                seasonal_order=(1,0,1,12),
                enforce_stationarity=False,
                enforce_invertibility=False)
results = model.fit()

In [ ]:
forecast = results.forecast(steps=len(test), exog=X_test)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, forecast))
mae = mean_absolute_error(y_test, forecast)
mape = np.mean(np.abs((y_test - forecast) / y_test)) * 100
model_accuracy = 100 - mape

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"MAPE: {mape:.2f}%")
print(f"Model Accuracy{model_accuracy:.2f}%")

In [ ]:
actual_df = kar_monthly['pm2_5'].reset_index()
actual_df['Type'] = 'Actual Data'

# Category 2: The predicted values over the test timeline
forecast_df = pd.DataFrame({
    'time': test.index,
    'pm2_5': forecast.values,
    'Type': 'Predicted (SARIMAX with Weather)'
})

# Combine them cleanly
plot_df = pd.concat([actual_df, forecast_df], ignore_index=True)

In [49]:

alt.data_transformers.disable_max_rows()

chart = alt.Chart(plot_df).mark_line().encode(
    x=alt.X('time:T', title='Date'),
    y=alt.Y('pm2_5:Q', title='PM2.5 Level'),
    color=alt.Color('Type:N', title='Legend', scale=alt.Scale(
        domain=['Actual Data', 'Predicted (SARIMAX with Weather)'],
        range=['#1f77b4', '#d62728'] # Blue for real data, Red for prediction
    )),
    strokeDash=alt.StrokeDash('Type:N', scale=alt.Scale(
        domain=['Actual Data', 'Predicted (SARIMAX with Weather)'],
        range=[[0], [4, 4]] # Solid line for actuals, dashed line for predictions
    )),
    tooltip=[alt.Tooltip('time:T', title='Date'), alt.Tooltip('pm2_5:Q', title='PM2.5')]
).properties(
    title='Accurate 10-Month PM2.5 Forecast using Weather Features',
    width=800,
    height=400
).interactive()

chart

C:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'


RMSE: 3.62
MAE: 2.57
MAPE: 7.54%


C:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


alt.Chart(...)

In [51]:
test_df = pd.DataFrame({
    'time': test.index,
    'pm2_5': np.asarray(test['pm2_5'] if 'pm2_5' in test else test).flatten(),
    'Type': 'Actual Test Data'
})

# 2. Structure the forecast data safely using .flatten()
forecast_df = pd.DataFrame({
    'time': test.index,
    'pm2_5': np.asarray(forecast).flatten(),
    'Type': 'Predicted Values'
})

# 3. Combine them into a single long-format DataFrame
plot_df = pd.concat([test_df, forecast_df], ignore_index=True)

# 4. Generate the Test vs Prediction Altair Chart
alt.data_transformers.disable_max_rows()

chart = alt.Chart(plot_df).mark_line(point=True).encode(
    x=alt.X('time:T', title='Date'),
    y=alt.Y('pm2_5:Q', title='PM2.5 Level'),
    color=alt.Color('Type:N', title='Legend', scale=alt.Scale(
        domain=['Actual Test Data', 'Predicted Values'],
        range=['#2ca02c', '#d62728']  # Green for Actual Test, Red for Predictions
    )),
    strokeDash=alt.StrokeDash('Type:N', scale=alt.Scale(
        domain=['Actual Test Data', 'Predicted Values'],
        range=[[0], [4, 4]]  # Solid line for actual test, dashed line for predictions
    )),
    tooltip=[alt.Tooltip('time:T', title='Date'), alt.Tooltip('pm2_5:Q', title='PM2.5')]
).properties(
    title='10-Month PM2.5: Actual Test Data vs Predicted Values',
    width=800,
    height=400
).interactive()

chart

alt.Chart(...)

In [ ]:
# ## For downlaod PKL file
# import joblib
# joblib.dump(results, 'sarimax_pm25_model.pkl')